## Association & Frequent-Pattern Mining
Reproduces basket-level association analysis (Apriori & FP-Growth), using `valid_purchase_lines` persisted from `preprocessing.ipynb`.

In [1]:
import pandas as pd

valid_purchase_lines = pd.read_csv("../data/processed/valid_purchase_lines.csv")
print(valid_purchase_lines.shape)
valid_purchase_lines[['InvoiceNo','StockCode','Description','Quantity']].head()

(349203, 15)


,InvoiceNo,StockCode,Description,Quantity
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6
1,536365,71053,WHITE METAL LANTERN,6
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6


In [2]:
# Stock codes that are NOT purely numeric-with-optional-letter-suffix (the normal product pattern)
# Normal: 85123, 85123A, 21730 etc. Administrative: POST, DOT, M, BANK CHARGES, AMAZONFEE, etc.
import re

def looks_like_product_code(code):
    return bool(re.match(r'^\d{4,6}[A-Za-z]?$', str(code)))

valid_purchase_lines['LooksLikeProduct'] = valid_purchase_lines['StockCode'].apply(looks_like_product_code)

non_product_codes = valid_purchase_lines[~valid_purchase_lines['LooksLikeProduct']]
print("Rows with non-standard stock codes:", non_product_codes.shape[0])
print(non_product_codes.groupby('StockCode')['Description'].first().sort_index())

Rows with non-standard stock codes: 560
StockCode
15056BL            EDWARDIAN PARASOL BLACK
BANK CHARGES                  Bank Charges
C2                                CARRIAGE
DOT                         DOTCOM POSTAGE
M                                   Manual
PADS            PADS TO MATCH ALL CUSHIONS
POST                               POSTAGE
Name: Description, dtype: object


In [3]:
# ---- Documented administrative code exclusion ----
# Confirmed non-product codes by manual inspection of descriptions:
ADMIN_CODES = ['BANK CHARGES', 'C2', 'DOT', 'M', 'POST']

print("Rows excluded as administrative:", valid_purchase_lines['StockCode'].isin(ADMIN_CODES).sum())

basket_source = valid_purchase_lines[~valid_purchase_lines['StockCode'].isin(ADMIN_CODES)].copy()
print("basket_source shape:", basket_source.shape)

Rows excluded as administrative: 328
basket_source shape: (348875, 16)


## Administrative Code Exclusion

Manually inspected 560 rows with non-standard StockCode patterns. Confirmed 5 codes as genuinely administrative (fees/postage, not purchasable products): `BANK CHARGES`, `C2` (carriage), `DOT` (dotcom postage), `M` (manual adjustment), `POST` (postage). Two initially-flagged codes (`15056BL`, `PADS`) were confirmed as real products via their descriptions and retained.

In [4]:
# ---- Step 1: distinct products per invoice (set, not list — dedupes automatically) ----
invoice_baskets_series = basket_source.groupby('InvoiceNo')['StockCode'].apply(lambda x: sorted(set(x)))

print("Number of invoices (baskets):", len(invoice_baskets_series))
print("Basket size distribution:")
print(invoice_baskets_series.apply(len).describe())

# quick look
invoice_baskets_series.head()

Number of invoices (baskets): 16579
Basket size distribution:
count    16579.000000
mean        20.756740
std         23.935496
min          1.000000
25%          6.000000
50%         15.000000
75%         27.000000
max        540.000000
Name: StockCode, dtype: float64


InvoiceNo
536365    [21730, 22752, 71053, 84029E, 84029G, 84406B, ...
536366                                       [22632, 22633]
536367    [21754, 21755, 21777, 22310, 22622, 22623, 227...
536368                         [22912, 22913, 22914, 22960]
536369                                              [21756]
Name: StockCode, dtype: object

In [5]:
from mlxtend.preprocessing import TransactionEncoder

baskets_list = invoice_baskets_series.tolist()

te = TransactionEncoder()
te_array = te.fit(baskets_list).transform(baskets_list)
basket_matrix = pd.DataFrame(te_array, columns=te.columns_, index=invoice_baskets_series.index)

print(basket_matrix.shape)
print("Memory usage (MB):", basket_matrix.memory_usage(deep=True).sum() / 1e6)

(16579, 3640)
Memory usage (MB): 60.480192


In [6]:
import os
os.makedirs("../data/processed", exist_ok=True)

# Save baskets as list-of-lists (compact) rather than the full one-hot matrix (huge/sparse)
invoice_baskets_df = invoice_baskets_series.reset_index()
invoice_baskets_df.columns = ['InvoiceNo', 'StockCodes']
invoice_baskets_df.to_csv("../data/processed/invoice_baskets.csv", index=False)
print("Saved invoice_baskets.csv")

Saved invoice_baskets.csv


In [7]:
from mlxtend.frequent_patterns import apriori
import time

support_candidates = [0.05, 0.03, 0.02, 0.015, 0.01]

sensitivity_results = []
for min_sup in support_candidates:
    try:
        start = time.time()
        freq_itemsets = apriori(basket_matrix, min_support=min_sup, use_colnames=True, low_memory=True)
        elapsed = time.time() - start
        sensitivity_results.append({
            'min_support': min_sup,
            'n_itemsets': len(freq_itemsets),
            'max_itemset_size': freq_itemsets['itemsets'].apply(len).max() if len(freq_itemsets) > 0 else 0,
            'time_seconds': round(elapsed, 2)
        })
        print(f"min_support={min_sup}: {len(freq_itemsets)} itemsets, {elapsed:.2f}s")
    except MemoryError:
        print(f"min_support={min_sup}: MemoryError — too low, stopping here")
        break

sensitivity_df = pd.DataFrame(sensitivity_results)
print(sensitivity_df)

min_support=0.05: 22 itemsets, 0.39s
min_support=0.03: 94 itemsets, 0.70s
min_support=0.02: 248 itemsets, 1.07s
min_support=0.015: 472 itemsets, 1.44s
min_support=0.01: 1051 itemsets, 2.19s
   min_support  n_itemsets  max_itemset_size  time_seconds
0        0.050          22                 1          0.39
1        0.030          94                 2          0.70
2        0.020         248                 3          1.07
3        0.015         472                 3          1.44
4        0.010        1051                 4          2.19


## Support Threshold Selection (Section 21)

Sensitivity analysis across 5 candidate thresholds on `basket_matrix` (16,579 invoices × 3,640 products):

| min_support | n_itemsets | max_itemset_size | time (s) |
|---|---|---|---|
| 0.05 | 22 | 1 | 0.39 |
| 0.03 | 94 | 2 | 0.70 |
| 0.02 | 248 | 3 | 1.07 |
| 0.015 | 472 | 3 | 1.44 |
| 0.01 | 1051 | 4 | 2.19 |

**Chosen threshold: min_support = 0.02.** At 0.05, itemsets are almost entirely single products (max size 1), unusable for association rules. 0.02 is the lowest threshold giving a manageable itemset count (248) while including meaningful multi-product combinations (size up to 3) — avoiding both the "too coarse to be useful" and "too many trivial rules" failure modes the blueprint warns against. min_support=0.01 (1,051 itemsets, size up to 4) is kept as a secondary threshold for extended comparison.

In [8]:
from mlxtend.frequent_patterns import association_rules

freq_itemsets_002 = apriori(basket_matrix, min_support=0.02, use_colnames=True, low_memory=True)

rules = association_rules(freq_itemsets_002, metric="confidence", min_threshold=0.3)
rules = rules.sort_values('lift', ascending=False)

print("Number of rules:", len(rules))
rules[['antecedents','consequents','support','confidence','lift']].head(15)

Number of rules: 76


,antecedents,consequents,support,confidence,lift
74,(22697),"(22698, 22699)",0.020568,0.557190,24.119179
71,"(22698, 22699)",(22697),0.020568,0.890339,24.119179
73,(22698),"(22697, 22699)",0.020568,0.691684,24.091222
72,"(22697, 22699)",(22698),0.020568,0.716387,24.091222
49,(22697),(22698),0.024368,0.660131,22.199406
48,(22698),(22697),0.024368,0.819473,22.199406
75,(22699),"(22698, 22697)",0.020568,0.502950,20.639618
70,"(22698, 22697)",(22699),0.020568,0.844059,20.639618
50,(22697),(22699),0.028711,0.777778,19.018846
51,(22699),(22697),0.028711,0.702065,19.018846


In [9]:
# Build a stable StockCode -> Description lookup (most common description per code, 
# since descriptions can vary slightly for the same code due to data entry)
code_to_desc = (
    valid_purchase_lines.groupby('StockCode')['Description']
    .agg(lambda x: x.mode()[0] if not x.mode().empty else x.iloc[0])
    .to_dict()
)

def describe_itemset(itemset):
    return [f"{code} ({code_to_desc.get(code, 'UNKNOWN')})" for code in itemset]

rules['antecedents_desc'] = rules['antecedents'].apply(lambda x: describe_itemset(list(x)))
rules['consequents_desc'] = rules['consequents'].apply(lambda x: describe_itemset(list(x)))

rules[['antecedents_desc','consequents_desc','support','confidence','lift']].head(15)

,antecedents_desc,consequents_desc,support,confidence,lift
74,[22697 (GREEN REGENCY TEACUP AND SAUCER)],"[22698 (PINK REGENCY TEACUP AND SAUCER), 22699...",0.020568,0.557190,24.119179
71,"[22698 (PINK REGENCY TEACUP AND SAUCER), 22699...",[22697 (GREEN REGENCY TEACUP AND SAUCER)],0.020568,0.890339,24.119179
73,[22698 (PINK REGENCY TEACUP AND SAUCER)],"[22697 (GREEN REGENCY TEACUP AND SAUCER), 2269...",0.020568,0.691684,24.091222
72,"[22697 (GREEN REGENCY TEACUP AND SAUCER), 2269...",[22698 (PINK REGENCY TEACUP AND SAUCER)],0.020568,0.716387,24.091222
49,[22697 (GREEN REGENCY TEACUP AND SAUCER)],[22698 (PINK REGENCY TEACUP AND SAUCER)],0.024368,0.660131,22.199406
48,[22698 (PINK REGENCY TEACUP AND SAUCER)],[22697 (GREEN REGENCY TEACUP AND SAUCER)],0.024368,0.819473,22.199406
75,[22699 (ROSES REGENCY TEACUP AND SAUCER )],"[22698 (PINK REGENCY TEACUP AND SAUCER), 22697...",0.020568,0.502950,20.639618
70,"[22698 (PINK REGENCY TEACUP AND SAUCER), 22697...",[22699 (ROSES REGENCY TEACUP AND SAUCER )],0.020568,0.844059,20.639618
50,[22697 (GREEN REGENCY TEACUP AND SAUCER)],[22699 (ROSES REGENCY TEACUP AND SAUCER )],0.028711,0.777778,19.018846
51,[22699 (ROSES REGENCY TEACUP AND SAUCER )],[22697 (GREEN REGENCY TEACUP AND SAUCER)],0.028711,0.702065,19.018846


In [10]:
# Keep only the highest-confidence rule for each unique combination of items involved
# (regardless of which side is antecedent vs consequent), to avoid counting the same
# underlying product relationship multiple times.
rules['item_group'] = rules.apply(
    lambda r: frozenset(r['antecedents']) | frozenset(r['consequents']), axis=1
)
rules_deduped = rules.sort_values('confidence', ascending=False).drop_duplicates(subset='item_group', keep='first')
rules_deduped = rules_deduped.sort_values('lift', ascending=False)

print("Rules before dedup:", len(rules), "| after dedup:", len(rules_deduped))
rules_deduped[['antecedents_desc','consequents_desc','support','confidence','lift']].head(15)

Rules before dedup: 76 | after dedup: 41


,antecedents_desc,consequents_desc,support,confidence,lift
71,"[22698 (PINK REGENCY TEACUP AND SAUCER), 22699...",[22697 (GREEN REGENCY TEACUP AND SAUCER)],0.020568,0.890339,24.119179
48,[22698 (PINK REGENCY TEACUP AND SAUCER)],[22697 (GREEN REGENCY TEACUP AND SAUCER)],0.024368,0.819473,22.199406
50,[22697 (GREEN REGENCY TEACUP AND SAUCER)],[22699 (ROSES REGENCY TEACUP AND SAUCER )],0.028711,0.777778,19.018846
52,[22698 (PINK REGENCY TEACUP AND SAUCER)],[22699 (ROSES REGENCY TEACUP AND SAUCER )],0.023102,0.776876,18.996802
64,[23300 (GARDENERS KNEELING PAD CUP OF TEA )],[23301 (GARDENERS KNEELING PAD KEEP CALM )],0.027625,0.730463,16.321210
54,[22726 (ALARM CLOCK BAKELIKE GREEN)],[22727 (ALARM CLOCK BAKELIKE RED )],0.027384,0.657971,14.391163
67,[82494L (WOODEN FRAME ANTIQUE WHITE )],[82482 (WOODEN PICTURE FRAME WHITE FINISH)],0.027625,0.584184,11.340962
35,[22910 (PAPER CHAIN KIT VINTAGE CHRISTMAS)],[22086 (PAPER CHAIN KIT 50'S CHRISTMAS )],0.026359,0.645495,11.324507
16,[20726 (LUNCH BAG WOODLAND)],[22382 (LUNCH BAG SPACEBOY DESIGN )],0.020870,0.513353,9.749005
57,[23202 (JUMBO BAG VINTAGE LEAF)],[23203 (JUMBO BAG VINTAGE DOILY )],0.024067,0.562764,9.501092


In [11]:
os.makedirs("../results/tables", exist_ok=True)
rules_deduped.drop(columns=['item_group']).to_csv("../results/tables/association_rules_apriori_sup002.csv", index=False)
print("Saved.")

Saved.


## Association Rules — Apriori (min_support=0.02)

76 raw rules generated (confidence ≥ 0.3), deduplicated to 41 unique item-relationships (removing directional/subset duplicates of the same underlying product group).

**Key finding:** the strongest associations (lift 8–24) are overwhelmingly same-product color/design variants (e.g., Regency Teacup and Saucer in Green/Pink/Roses; Bakelike Alarm Clock in Green/Red; multiple Lunch Bag designs), not cross-category associations. This suggests customers browsing this retailer's themed/decorative goods tend to buy multiple variants of the same item rather than combining unrelated product categories — a distinct and business-actionable insight (e.g., "customers who buy the Pink Regency Teacup are 82% likely to also buy the Green" → strong case for variant bundling/cross-display).

In [12]:
from mlxtend.frequent_patterns import fpgrowth
import time

# Apriori timing (already have this pattern, redo cleanly for fair comparison)
start = time.time()
freq_apriori = apriori(basket_matrix, min_support=0.02, use_colnames=True, low_memory=True)
time_apriori = time.time() - start

# FP-Growth timing
start = time.time()
freq_fpgrowth = fpgrowth(basket_matrix, min_support=0.02, use_colnames=True)
time_fpgrowth = time.time() - start

print(f"Apriori: {len(freq_apriori)} itemsets, {time_apriori:.3f}s")
print(f"FP-Growth: {len(freq_fpgrowth)} itemsets, {time_fpgrowth:.3f}s")

# Confirm they found the SAME itemsets (just possibly different order/format)
apriori_sets = set(freq_apriori['itemsets'])
fpgrowth_sets = set(freq_fpgrowth['itemsets'])
print("Same itemsets found:", apriori_sets == fpgrowth_sets)
print("In Apriori but not FP-Growth:", len(apriori_sets - fpgrowth_sets))
print("In FP-Growth but not Apriori:", len(fpgrowth_sets - apriori_sets))

Apriori: 248 itemsets, 1.210s
FP-Growth: 248 itemsets, 3.195s
Same itemsets found: True
In Apriori but not FP-Growth: 0
In FP-Growth but not Apriori: 0


## Apriori vs FP-Growth Comparison (Section 21, tasks 4-5)

| Algorithm | Itemsets found | Runtime (s) |
|---|---|---|
| Apriori | 248 | 1.21 |
| FP-Growth | 248 | 3.20 |

**Itemset agreement:** Identical — both algorithms found exactly the same 248 frequent itemsets at min_support=0.02 (0 discrepancies either direction), confirming both correctly implement the same frequent-itemset definition.

**Runtime:** Apriori was faster (1.21s vs 3.20s) at this scale. This is somewhat counter-intuitive relative to the common claim that FP-Growth is "generally faster" — that advantage typically comes from avoiding Apriori's candidate-generation cost on large, dense datasets with low support thresholds. At our scale (16,579 invoices, support=0.02, max itemset size 3), FP-Growth's tree-construction overhead outweighs the candidate-generation cost it's designed to avoid. The crossover point where FP-Growth wins would likely appear at lower support thresholds or larger basket counts.

In [13]:
cluster_assignments = pd.read_csv("../results/tables/rfm_cluster_assignments.csv")
cluster_assignments['CustomerID'] = cluster_assignments['CustomerID'].astype(str)

# Map each invoice to its customer's cluster
invoice_to_customer = valid_purchase_lines[['InvoiceNo','CustomerID']].drop_duplicates()
invoice_to_customer['CustomerID'] = invoice_to_customer['CustomerID'].astype(str)

invoice_to_cluster = invoice_to_customer.merge(
    cluster_assignments[['CustomerID','Cluster_k5']], on='CustomerID', how='inner'
)

print(invoice_to_cluster['Cluster_k5'].value_counts().sort_index())

Cluster_k5
0    6263
1    1271
2    4937
3    1302
4    2873
Name: count, dtype: int64


In [14]:
segment_baskets = {}
segment_freq_itemsets = {}
segment_rules = {}

for cluster_id in sorted(invoice_to_cluster['Cluster_k5'].unique()):
    invoices_in_cluster = invoice_to_cluster[invoice_to_cluster['Cluster_k5'] == cluster_id]['InvoiceNo']
    
    # Build baskets for just this segment
    seg_basket_series = invoice_baskets_df[invoice_baskets_df['InvoiceNo'].isin(invoices_in_cluster)].copy()
    seg_basket_series['StockCodes'] = seg_basket_series['StockCodes'].apply(
        lambda x: x if isinstance(x, list) else ast.literal_eval(x)
    )
    baskets_list_seg = seg_basket_series['StockCodes'].tolist()
    
    te_seg = TransactionEncoder()
    te_array_seg = te_seg.fit(baskets_list_seg).transform(baskets_list_seg)
    basket_matrix_seg = pd.DataFrame(te_array_seg, columns=te_seg.columns_)
    
    freq_seg = apriori(basket_matrix_seg, min_support=0.02, use_colnames=True, low_memory=True)
    rules_seg = association_rules(freq_seg, metric="confidence", min_threshold=0.3) if len(freq_seg) > 0 else pd.DataFrame()
    
    segment_baskets[cluster_id] = basket_matrix_seg
    segment_freq_itemsets[cluster_id] = freq_seg
    segment_rules[cluster_id] = rules_seg
    
    print(f"Cluster {cluster_id}: {len(invoices_in_cluster)} invoices, {len(freq_seg)} itemsets, {len(rules_seg)} rules")

Cluster 0: 6263 invoices, 323 itemsets, 203 rules
Cluster 1: 1271 invoices, 218 itemsets, 25 rules
Cluster 2: 4937 invoices, 313 itemsets, 115 rules
Cluster 3: 1302 invoices, 121 itemsets, 13 rules
Cluster 4: 2873 invoices, 271 itemsets, 79 rules


In [15]:
def dedupe_rules(rules_df):
    if len(rules_df) == 0:
        return rules_df
    r = rules_df.copy()
    r['item_group'] = r.apply(lambda row: frozenset(row['antecedents']) | frozenset(row['consequents']), axis=1)
    r = r.sort_values('confidence', ascending=False).drop_duplicates(subset='item_group', keep='first')
    return r.sort_values('lift', ascending=False)

segment_rules_deduped = {}
for cluster_id, rules_df in segment_rules.items():
    deduped = dedupe_rules(rules_df)
    segment_rules_deduped[cluster_id] = deduped
    print(f"Cluster {cluster_id}: {len(rules_df)} raw -> {len(deduped)} deduped")

# Top 5 rules per cluster, with descriptions, for side-by-side comparison
for cluster_id, deduped in segment_rules_deduped.items():
    print(f"\n=== Cluster {cluster_id} — top 5 rules by lift ===")
    if len(deduped) == 0:
        print("No rules.")
        continue
    top5 = deduped.head(5).copy()
    top5['antecedents_desc'] = top5['antecedents'].apply(lambda x: describe_itemset(list(x)))
    top5['consequents_desc'] = top5['consequents'].apply(lambda x: describe_itemset(list(x)))
    print(top5[['antecedents_desc','consequents_desc','support','confidence','lift']].to_string(index=False))

Cluster 0: 203 raw -> 107 deduped
Cluster 1: 25 raw -> 11 deduped
Cluster 2: 115 raw -> 58 deduped
Cluster 3: 13 raw -> 9 deduped
Cluster 4: 79 raw -> 42 deduped

=== Cluster 0 — top 5 rules by lift ===
                                                                 antecedents_desc                            consequents_desc  support  confidence      lift
                                         [22698 (PINK REGENCY TEACUP AND SAUCER)]   [22697 (GREEN REGENCY TEACUP AND SAUCER)] 0.026307    0.800000 20.109677
[22698 (PINK REGENCY TEACUP AND SAUCER), 22697 (GREEN REGENCY TEACUP AND SAUCER)]  [22699 (ROSES REGENCY TEACUP AND SAUCER )] 0.023260    0.884146 19.476213
                                        [22697 (GREEN REGENCY TEACUP AND SAUCER)]  [22699 (ROSES REGENCY TEACUP AND SAUCER )] 0.032563    0.818548 18.031204
                                         [22698 (PINK REGENCY TEACUP AND SAUCER)]  [22699 (ROSES REGENCY TEACUP AND SAUCER )] 0.026789    0.814634 17.944980
            

In [16]:
global_item_groups = set(rules_deduped['antecedents'].apply(frozenset)) | set(rules_deduped['consequents'].apply(frozenset))
# Better: use the item_group field we already built for global dedup
global_groups = set(rules_deduped['item_group'])

for cluster_id, deduped in segment_rules_deduped.items():
    if len(deduped) == 0:
        continue
    seg_groups = set(deduped['item_group'])
    unique_to_segment = seg_groups - global_groups
    print(f"Cluster {cluster_id}: {len(seg_groups)} unique item-groups, {len(unique_to_segment)} NOT present in global rules")

Cluster 0: 107 unique item-groups, 67 NOT present in global rules
Cluster 1: 11 unique item-groups, 9 NOT present in global rules
Cluster 2: 58 unique item-groups, 22 NOT present in global rules
Cluster 3: 9 unique item-groups, 4 NOT present in global rules
Cluster 4: 42 unique item-groups, 18 NOT present in global rules


**Finding 3 (quantified) — segment-specific rules invisible in global mining:**

| Cluster | Unique item-groups | Not in global rules | % segment-specific |
|---|---|---|---|
| 0 (highest value) | 107 | 67 | 62.6% |
| 1 | 11 | 9 | 81.8% |
| 2 | 58 | 22 | 37.9% |
| 3 (lowest value) | 9 | 4 | 44.4% |
| 4 | 42 | 18 | 42.9% |

Every segment shows a substantial share (38-82%) of rules absent from the global rule set — these relationships get diluted below the support threshold when all customers are pooled together. Cluster 1 is the most extreme case: 82% of its rules are undetectable globally. This provides strong, dataset-wide (not anecdotal) support for H5: segment-specific association rules meaningfully differ from global rules, and segmentation reveals genuine patterns that pooled analysis misses entirely.

In [17]:
os.makedirs("../results/tables/segment_rules", exist_ok=True)
for cluster_id, deduped in segment_rules_deduped.items():
    deduped.drop(columns=['item_group'], errors='ignore').to_csv(
        f"../results/tables/segment_rules/cluster{cluster_id}_rules.csv", index=False
    )
print("Saved all segment rule tables.")

Saved all segment rule tables.
